In [1]:
%pip install -qU langchain langchain-openai langchain-community pypdf

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_community.document_loaders import PyPDFLoader

documento = PyPDFLoader("documentos/Regras_26_27_PT_BR_52925fd6d0.pdf").load()

C:\Users\IATR\AppData\Local\Temp\ipykernel_19420\4086271058.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100
)

chunks = splitter.split_documents(documento)

In [11]:
chunks[80].page_content

'50\n5. Área de meta\n São traçadas duas linhas perpendiculares à linha de fundo, a 5,5 m de distância do \ninterior de cada trave. Essas linhas se prolongam para o interior do campo de jogo \npor 5,5 m e são unidas por uma linha paralela à linha de fundo. A área delimitada \npor essas linhas e pela linha de fundo constitui a área de meta.\n6. A área penal\n São traçadas duas linhas perpendiculares à linha de fundo, a 16,5 m de distância \ndo interior de cada trave. Essas linhas se prolongam para o interior do campo de \njogo por 16,5 m e são unidas por uma linha paralela à linha de fundo. A área \ndelimitada por essas linhas e pela linha de fundo constitui a área penal.\n Em cada área penal, a marca penal é marcada a 11 m de distância do ponto médio \nentre as traves.\n Um arco de círculo com um raio de 9,15 m a partir do centro de cada marca penal é \ntraçado no exterior da área penal.\n7. A área do escanteio\n A área do escanteio ou área de tiro de canto é delimitada por um quarto d

In [14]:
from openai import OpenAI
from os import getenv

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=getenv("OPENROUTER_API_KEY")
)

chunk_texts = [chunk.page_content for chunk in chunks]
batch_size = 256
all_embeddings = []

for start in range(0, len(chunk_texts), batch_size):
    batch = chunk_texts[start:start + batch_size]
    response = client.embeddings.create(
        model="nvidia/nemotron-3-embed-1b:free",
        input=batch,
        encoding_format="float",
    )
    all_embeddings.extend(response.data)

print(f"Created {len(all_embeddings)} embeddings")
print(all_embeddings[0].embedding[:10])

Created 449 embeddings
[0.11317405, 0.01420262, -0.017303111, -0.02345115, 0.0027289246, -0.005477337, -0.009875056, 0.027367847, 0.04879975, -0.0044496865]


In [22]:
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_core.embeddings import Embeddings

class OpenRouterEmbeddings(Embeddings):
    def embed_documents(self, texts):
        texts = [text for text in texts if isinstance(text, str) and text.strip()]
        if not texts:
            return []

        batch_size = 256
        all_embeddings = []
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            response = client.embeddings.create(
                model="nvidia/nemotron-3-embed-1b:free",
                input=batch,
                encoding_format="float",
            )
            data = getattr(response, "data", None) or []
            all_embeddings.extend([item.embedding for item in data if getattr(item, "embedding", None) is not None])
        return all_embeddings

    def embed_query(self, text):
        response = client.embeddings.create(
            model="nvidia/nemotron-3-embed-1b:free",
            input=[text],
            encoding_format="float",
        )
        data = getattr(response, "data", None) or []
        if not data:
            return []
        return data[0].embedding

embeddings = OpenRouterEmbeddings()

valid_chunks = [chunk for chunk in chunks if getattr(chunk, "page_content", "").strip()]
vectorstore = InMemoryVectorStore.from_documents(
    documents=valid_chunks,
    embedding=embeddings
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [25]:
retriever.invoke("Impedimento")

[Document(id='7a8b379f-3fd0-4a92-ba45-bd34af161ffa', metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.4 (Windows)', 'creationdate': '2026-06-11T12:05:39+02:00', 'moddate': '2026-06-11T12:06:16+02:00', 'trapped': '/False', 'source': 'documentos/Regras_26_27_PT_BR_52925fd6d0.pdf', 'total_pages': 268, 'page': 114, 'page_label': '115'}, page_content='2. Infração por impedimento\n Um jogador em posição de impedimento quando a bola for tocada* por um \ncompanheiro de equipe será punido somente se chegar a participar do jogo de \nforma ativa, ao:\n•\u2002 interferir no jogo por tocar em uma bola passada ou tocada por um \ncompanheiro; ou\n•\u2002interferir em um adversário ao:\n\u2002•  impedi-lo de jogar ou ter condições de jogar a bola por claramente obstruir o \ncampo de visão do adversário; ou \n\u2002• disputar a bola com o adversário; ou\n * Deverá ser considerado o primeiro ponto de contato ao tocar na bola; porém, \nquando a bola for arremessada pelo golei

In [26]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "Você é um especialista nas regras do futebol. "
    "Responda a pergunta usando APENAS o contexto abaixo. "
    "Se a resposta não estiver no contexto, diga que não encontrou a informação.\n\n"
    "Contexto:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

In [27]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [29]:
rag_chain.invoke("Como funciona o impedimento no futebol?")

'De acordo com o contexto fornecido, o funcionamento do impedimento divide-se em posição de impedimento e infração por impedimento:\n\n**1. Posição de impedimento**\nEstar em posição de impedimento não é uma infração. Um jogador estará em posição de impedimento se:\n*   Qualquer parte de sua cabeça, seu corpo ou seus pés estiver no campo do adversário (excluindo a linha de meio de campo); **e**\n*   Qualquer parte de sua cabeça, seu corpo ou seus pés estiver mais próxima da linha de fundo do campo do adversário do que a bola e o penúltimo adversário.\n\n**Observações sobre a posição:**\n*   Não são considerados as mãos nem os braços de nenhum jogador (incluindo goleiros); o limite superior do braço é definido pelo ponto inferior da axila.\n*   O jogador **não** estará em posição de impedimento se estiver na mesma linha do penúltimo adversário ou dos dois últimos adversários.\n\n**2. Infração por impedimento**\nUm jogador em posição de impedimento, quando a bola for tocada por um compan